[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S09/2_Practica/en_clase/S09_04_q_learning_en_clase.ipynb)

# S09 · 04 — Aprendizaje por refuerzo: Q-learning en el hotel · versión en clase

⏱ 25 min en clase · Sprint 9 · M3 (UD 3.12, introducción al aprendizaje por refuerzo)

**Contexto TurisData.** En el aprendizaje supervisado alguien nos da la respuesta correcta. Aquí **no**: un **agente** prueba acciones, recibe **recompensas** (o castigos) y aprende por ensayo y error una **política** (qué hacer en cada situación). Vamos a entrenar a un "botones virtual" que debe llevar a un huésped desde recepción hasta la habitación con vistas al mar, evitando la zona en obras. Es un juguete, pero contiene las piezas de cualquier problema de refuerzo (robots, juegos, precios dinámicos).

**Este cuaderno tiene 1 ejercicio**, el que se hace en el aula. La función epsilon-greedy (en clase te la damos hecha), leer la política aprendida, el experimento sin exploración, el mini-reto de cambiar las reglas del negocio y las preguntas de reflexión están en el cuaderno completo. Su ejercicio 1 es el de este.

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEMILLA = 42
np.random.seed(SEMILLA)
sns.set_theme(style="white")
print("Listo ✔")

### 👀 A2 · EJEMPLO — El entorno del hotel y tres pasos de prueba

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Qué estado y qué recompensa devuelve el primer paso a la derecha?
2. En el tercer paso (abajo), ¿por qué el estado no cambia y aun así se paga −1?
3. ¿Qué recompensa total tiene llegar a la meta sin pasar por las obras en 7 pasos?

In [ ]:
class EntornoHotel:
    """Cuadrícula sencilla con la misma interfaz que Gymnasium: reset() y step(accion)."""
    FILAS, COLUMNAS = 4, 5
    N_ESTADOS, N_ACCIONES = 20, 4
    PAREDES = {(1, 1), (1, 2), (3, 1)}
    OBRAS = (1, 4)
    META = (3, 4)
    MOVIMIENTOS = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
    NOMBRES_ACCION = ["arriba", "abajo", "izquierda", "derecha"]

    def __init__(self, coste_paso=1.0):
        self.coste_paso = coste_paso      # lo que "cuesta" cada paso (1.0 por defecto)

    def reset(self):
        self.pos = (0, 0)
        return self.estado()

    def estado(self):
        return self.pos[0] * self.COLUMNAS + self.pos[1]

    def step(self, accion):
        df, dc = self.MOVIMIENTOS[accion]
        nueva = (self.pos[0] + df, self.pos[1] + dc)
        dentro = 0 <= nueva[0] < self.FILAS and 0 <= nueva[1] < self.COLUMNAS
        if dentro and nueva not in self.PAREDES:
            self.pos = nueva
        recompensa = -self.coste_paso
        if self.pos == self.OBRAS:
            recompensa -= 5.0
        terminado = self.pos == self.META
        if terminado:
            recompensa += 10.0
        return self.estado(), recompensa, terminado


entorno = EntornoHotel()
s = entorno.reset()
print("Estado inicial:", s)
for a in (3, 3, 1):   # derecha, derecha, abajo (choca con la pared)
    s, r, fin = entorno.step(a)
    print(f"acción {EntornoHotel.NOMBRES_ACCION[a]:>9} → estado {s:>2}, recompensa {r:+.0f}, fin={fin}")

### ✏️ A3 · EJERCICIO 1 de 1 — La regla de actualización de Q

**Tu turno:** Escribe `actualizar_q(Q, s, a, r, s2, alpha, gamma)` que aplique la regla **modificando `Q` en su sitio** (no hace falta devolver nada).

**Debe salir:** ✔ Ejercicio 1 correcto (con `Q[0,3]=0`, `r=-1`, `max Q[5]=4`, `alpha=0.5` y `gamma=0.9` sale `Q[0,3] = 1.3`)

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A3).

1. ¿Qué valor queda en `Q[0, 3]` y cómo sale?
2. ¿Qué pasaría con `alpha = 1`?
3. ¿Qué pasaría con `gamma = 0`?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A3, **después de intentarlo**.

In [ ]:
def actualizar_q(Q, s, a, r, s2, alpha, gamma):
    # TODO: Primero calcula lo que ahora crees que vale la acción (recompensa + futuro descontado); luego mueve Q[s, a] una fracción alpha hacia ese valor
    ...

Qprueba = np.zeros((20, 4))
Qprueba[5, 3] = 4.0
actualizar_q(Qprueba, s=0, a=3, r=-1.0, s2=5, alpha=0.5, gamma=0.9)
assert abs(Qprueba[0, 3] - 1.3) < 1e-9, "Con Q[0,3]=0, r=-1, max Q[5]=4, alpha=0.5, gamma=0.9 debe salir 0.5·(-1+0.9·4)=1.3"
assert np.count_nonzero(Qprueba) == 2, "Solo debes modificar Q[s, a]"
print("✔ Ejercicio 1 correcto")

### 👀 A4 · EJEMPLO — Elegir acción con epsilon-greedy (ya hecho)

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A4).

1. Con `epsilon = 0`, ¿qué acción elige y por qué?
2. Con `epsilon = 1`, ¿qué ocurre?
3. ¿Qué riesgo tiene un agente que nunca explora (`epsilon = 0`)?

In [ ]:
def elegir_accion(Q, s, epsilon, rng):
    if rng.random() < epsilon:
        return int(rng.integers(Q.shape[1]))
    return int(np.argmax(Q[s]))

rng_t = np.random.default_rng(0)
Qt = np.zeros((20, 4)); Qt[7] = [0, 0, 5, 0]
print("epsilon = 0 → acción elegida:", elegir_accion(Qt, 7, 0.0, rng_t), "(la de mayor valor)")
print("epsilon = 1 → acciones vistas en 200 intentos:", sorted({elegir_accion(Qt, 7, 1.0, rng_t) for _ in range(200)}))

### 👀 A5 · EJEMPLO — Entrenar al agente con tus dos funciones

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A5).

1. ¿Cuál es la recompensa media al principio y al final del entrenamiento?
2. ¿Por qué la recompensa de los últimos episodios (0,9) no llega a +3, el máximo posible?
3. ¿Qué guarda la tabla `Q` al final?

In [ ]:
def entrenar(episodios=1500, alpha=0.2, gamma=0.95, epsilon=0.2, semilla=SEMILLA, max_pasos=50, coste_paso=1.0):
    rng = np.random.default_rng(semilla)
    env = EntornoHotel(coste_paso)
    Q = np.zeros((env.N_ESTADOS, env.N_ACCIONES))
    recompensas = []
    for _ in range(episodios):
        s = env.reset()
        total = 0.0
        for _ in range(max_pasos):
            a = elegir_accion(Q, s, epsilon, rng)
            s2, r, fin = env.step(a)
            actualizar_q(Q, s, a, r, s2, alpha, gamma)
            total += r
            s = s2
            if fin:
                break
        recompensas.append(total)
    return Q, np.array(recompensas)

Q, recompensas = entrenar()
print("Recompensa media de los primeros 50 episodios:", recompensas[:50].mean().round(1))
print("Recompensa media de los últimos 50 episodios: ", recompensas[-50:].mean().round(1))

### 👀 A6 · EJEMPLO — La curva de aprendizaje

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A6).

1. ¿Cuántos episodios tarda aproximadamente en dejar de perder recompensa?
2. ¿Por qué la línea suave se mantiene sobre 0–2 y no sigue subiendo?
3. ¿Por qué se dibuja la media móvil además de cada episodio?

In [ ]:
suave = pd.Series(recompensas).rolling(30).mean()
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(recompensas, alpha=0.25, label="cada episodio")
ax.plot(suave, color="tab:red", label="media móvil (30)")
ax.set_xlabel("Episodio"); ax.set_ylabel("Recompensa total")
ax.set_title("El agente mejora con la experiencia"); ax.legend()
sns.despine(); plt.show()

---

## 🏁 Fin del cuaderno de clase

El **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S09/2_Practica/para_casa/S09_04_q_learning.ipynb)** trae la función epsilon-greedy, la lectura de la política aprendida, el experimento sin exploración, el mini-reto de cambiar las reglas del negocio y las preguntas de reflexión, cada ejercicio con su solución plegada en la Parte B.

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas

- **Qué hace:**
    - Carga las herramientas: `numpy`, `pandas`, `matplotlib` y `seaborn` (todo el cuaderno está hecho con numpy, sin librerías de refuerzo).
    - `dato(...)` es la función de descarga de datos del curso (en este cuaderno no hace falta, no hay datos externos).
    - Fija la semilla `SEMILLA = 42` para que los entrenamientos den siempre lo mismo.
- **Qué debe salir:**
    - `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'np' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 1. El entorno: el pasillo del hotel

#### A2 · Ejemplo — El entorno del hotel y tres pasos de prueba

**Contexto TurisData.** En el aprendizaje supervisado alguien nos da la respuesta correcta. Aquí **no**: un **agente** prueba acciones, recibe **recompensas** (o castigos) y aprende por ensayo y error una **política** (qué hacer en cada situación). Vamos a entrenar a un «botones virtual» que debe llevar a un huésped desde recepción hasta la habitación con vistas al mar, evitando la zona en obras.

Una cuadrícula de 4 filas × 5 columnas:

```
  S  .  .  .  .        S = recepción (inicio)
  .  #  #  .  X        # = pared (no se puede cruzar)
  .  .  .  .  .        X = zona en obras (recompensa -5, pero se puede pisar)
  .  #  .  .  G        G = habitación con vistas al mar (recompensa +10, fin del episodio)
```

- **Estado:** la casilla donde está el agente (20 posibles, numeradas de 0 a 19 por filas).
- **Acciones:** 0 = arriba, 1 = abajo, 2 = izquierda, 3 = derecha.
- **Recompensa:** -1 por cada paso (el huésped tiene prisa), -5 extra en obras, +10 al llegar a G.
- Si el agente choca con una pared o el borde, **se queda donde está** (y paga el paso).
- **Episodio:** una partida, desde recepción hasta llegar a G (o hasta 50 pasos).

- **Qué hace:**
    - La clase `EntornoHotel` es el mundo del agente, con la misma interfaz que Gymnasium: `reset()` lo coloca en recepción y devuelve el estado inicial; `step(accion)` mueve al agente y devuelve `(estado nuevo, recompensa, ¿terminado?)`.
    - Si el movimiento sale de la cuadrícula o choca con una pared, el agente **se queda donde está** (y paga el paso). Pisar las obras cuesta −5 extra y llegar a la meta da +10.
    - El código final crea un entorno y prueba tres acciones: derecha, derecha y abajo (esta última choca con una pared).
- **Qué debe salir:**
    - `Estado inicial: 0` y tres líneas: `acción derecha → estado 1, recompensa -1, fin=False`, `acción derecha → estado 2, recompensa -1, fin=False` y `acción abajo → estado 2, recompensa -1, fin=False`.
- **Cómo leerlo:**
    - La tercera línea muestra el choque: el estado sigue siendo el 2 y aun así cuesta −1.
- **Si falla:**
    - `SyntaxError`/`IndentationError` → se ha copiado mal la clase: ejecútala tal cual está.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué estado y qué recompensa devuelve el primer paso a la derecha?* → Estado **1** y recompensa **−1** (el coste de cada paso).
2. *En el tercer paso (abajo), ¿por qué el estado no cambia y aun así se paga −1?* → Porque abajo hay una **pared**: el agente se queda donde está, pero el paso cuesta igual.
3. *¿Qué recompensa total tiene llegar a la meta sin pasar por las obras en 7 pasos?* → −1 × 7 + 10 = **+3**: es el máximo posible (el «+3 óptimo»).

</details>

## 2. La tabla Q y su regla de actualización

#### A3 · Ejercicio 1 — La regla de actualización de Q

**Idea.** Guardamos una tabla `Q[estado, acción]` con *"cuánta recompensa total espero si estoy aquí y
hago esta acción y luego sigo lo mejor posible"*. Empieza en ceros. Tras cada paso se corrige un poco:

`Q[s,a] ← Q[s,a] + alpha · ( r + gamma · max(Q[s',:]) − Q[s,a] )`

- `alpha` (0–1): **velocidad de aprendizaje**. Alto = se cree lo último; bajo = cambia despacio.
- `gamma` (0–1): cuánto **importa el futuro**. Con 0 el agente solo mira la recompensa inmediata.
- `r + gamma·max(Q[s'])` es lo que **ahora** creemos que vale esa acción; se compara con lo que creíamos antes.

- **Qué hace:**
    - `objetivo = r + gamma * Q[s2].max()` es lo que **ahora** creemos que vale la acción: la recompensa inmediata más el valor descontado de la mejor opción desde el estado siguiente.
    - `Q[s, a] += alpha * (objetivo - Q[s, a])` mueve el valor antiguo una fracción `alpha` hacia el objetivo. No devuelve nada: modifica `Q` en su sitio.
    - La comprobación crea una tabla de prueba con `Q[5, 3] = 4`, aplica la regla con `alpha=0.5` y `gamma=0.9` y verifica `Q[0, 3] = 0.5 · (−1 + 0.9 · 4) = 1.3` y que solo cambia esa casilla.
- **Qué debe salir:**
    - `✔ Ejercicio 1 correcto`.
- **Si falla:**
    - `Con Q[0,3]=0, r=-1, max Q[5]=4, alpha=0.5, gamma=0.9 debe salir 0.5·(-1+0.9·4)=1.3` → la fórmula no es la de la regla (¿falta `alpha`, el `max` de `Q[s2]` o el `- Q[s, a]`?).
    - `Solo debes modificar Q[s, a]` → has cambiado otras casillas de `Q` (por ejemplo, la fila entera).
    - `NameError: name 'np' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué valor queda en `Q[0, 3]` y cómo sale?* → **1,3**: objetivo = −1 + 0,9 × 4 = 2,6 y se avanza la mitad (`alpha = 0,5`) desde 0: 0,5 × 2,6.
2. *¿Qué pasaría con `alpha = 1`?* → Que `Q[s, a]` se quedaría **directamente** con el objetivo (2,6): se cree lo último que ha visto, sin memoria de lo anterior.
3. *¿Qué pasaría con `gamma = 0`?* → El objetivo sería solo la recompensa inmediata (−1): el agente **no mira al futuro** y no puede aprender a llegar a una meta lejana.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def actualizar_q(Q, s, a, r, s2, alpha, gamma):
    objetivo = r + gamma * Q[s2].max()
    Q[s, a] += alpha * (objetivo - Q[s, a])
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 3. Explorar o explotar: epsilon-greedy

#### A4 · Ejemplo — Elegir acción con epsilon-greedy (ya hecho)

**Idea.** Si el agente siempre hace lo que cree mejor, nunca descubre caminos mejores. Si siempre
experimenta, nunca aprovecha lo aprendido. **Epsilon-greedy**: con probabilidad `epsilon` hace una acción
**al azar** (explorar) y con probabilidad `1 - epsilon` la **mejor conocida** (explotar).

En la versión completa escribir esta función es el ejercicio 2; en clase te la damos hecha porque el entrenamiento la necesita.

- **Qué hace:**
    - `elegir_accion(Q, s, epsilon, rng)`: `rng.random() < epsilon` decide si explorar; si sí, devuelve una acción al azar (`rng.integers`); si no, la de mayor valor de la fila `Q[s]` (`np.argmax`).
    - El código de prueba usa una tabla donde en el estado 7 la mejor acción es la 2 (izquierda): con `epsilon = 0` elige siempre la 2, y con `epsilon = 1` prueba las cuatro.
- **Qué debe salir:**
    - `epsilon = 0 → acción elegida: 2 (la de mayor valor)` y `epsilon = 1 → acciones vistas en 200 intentos: [0, 1, 2, 3]`.
- **Si falla:**
    - `NameError: name 'np' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *Con `epsilon = 0`, ¿qué acción elige y por qué?* → La **2**: es la de mayor valor en `Qt[7]`; sin exploración siempre explota.
2. *Con `epsilon = 1`, ¿qué ocurre?* → Elige **al azar** entre las cuatro acciones: explora siempre.
3. *¿Qué riesgo tiene un agente que nunca explora (`epsilon = 0`)?* → Que **no descubre caminos mejores**: se queda con lo primero que le pareció bueno.

</details>

## 4. Entrenar al agente

#### A5 · Ejemplo — Entrenar al agente con tus dos funciones

Un bucle: en cada episodio, el agente empieza en recepción, elige acciones, recibe recompensas y
actualiza `Q` con **tus** dos funciones. Guardamos la recompensa total de cada episodio para ver si mejora.

- **Qué hace:**
    - `entrenar(...)` repite 1 500 episodios: el agente empieza en recepción (`env.reset()`), y en cada paso elige una acción (`elegir_accion`), la ejecuta (`env.step`) y actualiza la tabla (`actualizar_q`), hasta llegar a la meta o agotar 50 pasos.
    - Guarda la recompensa total de cada episodio (`recompensas`) y devuelve la tabla `Q` y esas recompensas.
    - `Q, recompensas = entrenar()` entrena con los valores por defecto (`alpha=0.2`, `gamma=0.95`, `epsilon=0.2`); los dos `print` comparan los primeros y los últimos 50 episodios.
- **Qué debe salir:**
    - `Recompensa media de los primeros 50 episodios: -8.0` y `Recompensa media de los últimos 50 episodios:  0.9`.
- **Cómo leerlo:**
    - La recompensa media pasa de −8,0 a 0,9: el agente ha aprendido. No llega a +3 (el óptimo) porque con `epsilon = 0,2` sigue dando pasos al azar.
- **Si falla:**
    - `NameError: name 'actualizar_q' is not defined` (o `elegir_accion`) → falta el ejercicio 1 (o el ejemplo «ya hecho» de epsilon-greedy).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuál es la recompensa media al principio y al final del entrenamiento?* → **−8,0** en los primeros 50 episodios y **0,9** en los últimos 50.
2. *¿Por qué la recompensa de los últimos episodios (0,9) no llega a +3, el máximo posible?* → Porque con `epsilon = 0,2` el 20 % de las acciones siguen siendo al azar durante el entrenamiento: a veces se pierde algún paso o se pisan las obras.
3. *¿Qué guarda la tabla `Q` al final?* → Para cada estado y acción, la **recompensa total esperada** si se hace esa acción y luego se sigue lo mejor posible.

</details>

#### A6 · Ejemplo — La curva de aprendizaje

- **Qué hace:**
    - `pd.Series(recompensas).rolling(30).mean()` calcula la media móvil de 30 episodios (`suave`).
    - El gráfico dibuja la recompensa de cada episodio (línea tenue, muy ruidosa) y la media móvil (línea roja).
- **Qué debe salir:**
    - Una curva de aprendizaje: la media móvil empieza en torno a −13 en los primeros episodios, sube a ≈ 0 en unas decenas de episodios y después se mantiene entre 0 y 2 con altibajos.
- **Cómo leerlo:**
    - Lo importante es que **sube deprisa y se estabiliza**. Los picos negativos aislados son episodios en los que la exploración al azar lleva al agente a pisar las obras o a dar muchas vueltas.
- **Si falla:**
    - `NameError: name 'recompensas' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántos episodios tarda aproximadamente en dejar de perder recompensa?* → Unos **30–60 episodios**: la media móvil pasa de ≈ −13 a ≈ 0.
2. *¿Por qué la línea suave se mantiene sobre 0–2 y no sigue subiendo?* → Porque el agente ya conoce el camino casi óptimo (+3); lo que queda es el efecto de explorar con `epsilon = 0,2`.
3. *¿Por qué se dibuja la media móvil además de cada episodio?* → Porque cada episodio es **muy ruidoso** (depende del azar); la media móvil deja ver la tendencia.

</details>